# Build Your First Agent

In this notebook, you'll build AI agents step by step — from a simple conversational agent to one that connects to SQL Server and retrieves real data.

| # | What You'll Build | Key Concept |
|---|---|---|
| 1 | Hello Agent | 3 lines of code |
| 2 | Database Expert | System prompts |
| 3 | SQL Server Agent | `@tool` decorator |
| 4 | Multi-tool Agent | Multiple tools + reasoning |

---
## Setup

Verify Strands SDK is installed and check which model we're using:

In [ ]:
# Verify installation
import strands
print(f"✅ Strands SDK ready")

# Check default model
from strands import Agent
agent = Agent()
print(f"Default model: {agent.model.config}")

---
## 1. Hello Agent

It takes just 3 lines — import, create, invoke:

```python
from strands import Agent       # 1 - Import
agent = Agent()                 # 2 - Create (defaults to Bedrock Claude Sonnet)
response = agent("question")   # 3 - Invoke
```

💡 No tools, no configuration — the agent uses its built-in knowledge to answer.

In [ ]:
from strands import Agent

agent = Agent()
response = agent("What is Amazon RDS for SQL Server? Keep it to 3 sentences.")
print(response)

---
## 2. Database Expert Agent (System Prompt)

The `system_prompt` defines the agent's personality and expertise. It's the instruction that shapes every response.

🔑 In this workshop, each agent has a system prompt that defines its specialty — health monitoring, query performance, security, etc.

In [ ]:
from strands import Agent
from strands.models import BedrockModel
import os

model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region_name=os.getenv("AWS_REGION", "us-west-2"),
)

agent = Agent(
    model=model,
    system_prompt="""You are a helpful AWS database expert assistant.
    
You specialize in:
- Amazon RDS for SQL Server performance and optimization
- Database monitoring and troubleshooting
- Wait events, blocking, and query tuning
- Best practices for production database operations

Keep answers concise and actionable."""
)

response = agent("What are the top 3 things to check when SQL Server CPU is high?")
print(response)

In [ ]:
# The agent remembers context from the previous turn
response = agent("Can you elaborate on the first one?")
print(response)

---
## 3. SQL Server Agent with Tools

Agents become powerful when you give them **tools** — functions they can call to interact with the real world.

```
You ask a question
       ↓
Agent REASONS about what to do
       ↓
Agent CALLS the right tool
       ↓
Tool RETURNS data
       ↓
Agent RESPONDS with analysis
```

🔑 This is the pattern used throughout the workshop — every database diagnostic query becomes a `@tool`.

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel
import boto3
import pymssql
import json
import os

@tool
def get_sqlserver_version():
    """Connect to SQL Server and retrieve the version information."""
    client = boto3.client('secretsmanager', region_name=os.getenv('AWS_REGION', 'us-west-2'))
    secret = client.get_secret_value(SecretId='dbops-infra-sqlserver-secret')
    creds = json.loads(secret['SecretString'])

    conn = pymssql.connect(
        server=creds['host'],
        user=creds['username'],
        password=creds['password'],
        port=creds['port']
    )
    cursor = conn.cursor()
    cursor.execute("SELECT @@VERSION")
    version = cursor.fetchone()[0]
    cursor.close()
    conn.close()
    return version

model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region_name=os.getenv("AWS_REGION", "us-west-2"),
)

agent = Agent(
    model=model,
    system_prompt="You are a SQL Server assistant. Use your tools to answer questions about the database.",
    tools=[get_sqlserver_version]
)

response = agent("What version of SQL Server are we running?")
print(response)

---
## 4. Multi-Tool Agent

Real agents have multiple tools. The agent **reasons** about which tool to call based on your question.

Let's add a tool that lists databases and another that counts tables:

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel
import boto3, pymssql, json, os

def _get_connection():
    """Helper to get SQL Server connection from Secrets Manager."""
    client = boto3.client('secretsmanager', region_name=os.getenv('AWS_REGION', 'us-west-2'))
    secret = client.get_secret_value(SecretId='dbops-infra-sqlserver-secret')
    creds = json.loads(secret['SecretString'])
    return pymssql.connect(
        server=creds['host'], user=creds['username'],
        password=creds['password'], port=creds['port']
    )

@tool
def list_databases():
    """List all user databases on the SQL Server instance."""
    conn = _get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sys.databases WHERE database_id > 4 ORDER BY name")
    databases = [row[0] for row in cursor.fetchall()]
    cursor.close()
    conn.close()
    return databases

@tool
def get_table_count(database_name: str):
    """Get the number of user tables in a specific database."""
    conn = _get_connection()
    cursor = conn.cursor()
    cursor.execute(f"SELECT COUNT(*) FROM [{database_name}].sys.tables")
    count = cursor.fetchone()[0]
    cursor.close()
    conn.close()
    return f"{database_name} has {count} tables"

@tool
def get_stored_procedures(database_name: str):
    """List stored procedures in a database."""
    conn = _get_connection()
    cursor = conn.cursor()
    cursor.execute(f"SELECT name FROM [{database_name}].sys.procedures WHERE is_ms_shipped = 0 ORDER BY name")
    procs = [row[0] for row in cursor.fetchall()]
    cursor.close()
    conn.close()
    return procs

model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region_name=os.getenv("AWS_REGION", "us-west-2"),
)

agent = Agent(
    model=model,
    system_prompt="You are a SQL Server DBA assistant. Use your tools to explore the database.",
    tools=[list_databases, get_table_count, get_stored_procedures]
)

# The agent decides WHICH tools to call based on the question
response = agent("What databases exist and how many tables does DBOpsLab have?")
print(response)

In [ ]:
# Ask about stored procedures — agent picks the right tool
response = agent("What stored procedures are in DBOpsLab?")
print(response)

---
## Key Takeaways

| Concept | Code |
|---|---|
| Create an agent | `agent = Agent()` |
| Set behavior | `system_prompt="..."` |
| Add tools | `@tool` decorator + `tools=[...]` |
| Agent reasons | Picks which tool(s) to call based on your question |
| Multi-turn | Same agent instance remembers conversation |

**Next:** You'll explore the 5 specialized database agents that use these exact patterns with 69 tools across CloudWatch, Database Insights, DMVs, and RDS APIs.